# Before your start:
- Read the README.md file
- Comment as much as you can and use the resources (README.md file)
- Happy learning!

In [1]:
# import numpy and pandas
import pandas as pd
import numpy as np
import scipy.stats as st
from scipy.stats import trim_mean, mode, skew, gaussian_kde, pearsonr, spearmanr, beta
from statsmodels.stats.weightstats import ztest as ztest
from statsmodels.stats.proportion import proportions_ztest

from scipy.stats import ttest_ind, norm, t
from scipy.stats import f_oneway
from scipy.stats import sem

# Challenge 1 - Exploring the Data

In this challenge, we will examine all salaries of employees of the City of Chicago. We will start by loading the dataset and examining its contents

In [2]:
# Run this code:
salaries = pd.read_csv('../data/Current_Employee_Names__Salaries__and_Position_Titles.csv')

Examine the `salaries` dataset using the `head` function below.

In [3]:
# Your code here
salaries.head()

,Name,Job Titles,Department,Full or Part-Time,Salary or Hourly,Typical Hours,Annual Salary,Hourly Rate
0,"AARON, JEFFERY M",SERGEANT,POLICE,F,Salary,NaN,101442.0,NaN
1,"AARON, KARINA",POLICE OFFICER (ASSIGNED AS DETECTIVE),POLICE,F,Salary,NaN,94122.0,NaN
2,"AARON, KIMBERLEI R",CHIEF CONTRACT EXPEDITER,GENERAL SERVICES,F,Salary,NaN,101592.0,NaN
3,"ABAD JR, VICENTE M",CIVIL ENGINEER IV,WATER MGMNT,F,Salary,NaN,110064.0,NaN
4,"ABASCAL, REECE E",TRAFFIC CONTROL AIDE-HOURLY,OEMC,P,Hourly,20.0,NaN,19.86


Every row is one employee. Salaried employees only have an `Annual Salary` and hourly workers only have an `Hourly Rate`. So for the next challenges I first have to filter for the 8,022 hourly workers (out of 33,183 employees). Their `Hourly Rate` has no missing values, so nothing has to be dropped.

# Challenge 2 - Hypothesis Tests

In this section of the lab, we will test whether the hourly wage of all hourly workers is significantly different from $30/hr. Use only the hourly workers (`Salary or Hourly` is `Hourly`) and their `Hourly Rate`. Import the correct one sample test function from scipy and perform a two-sided test at the 5% significance level. Write down H0 and H1 first, then state your conclusion.

First the hypotheses, at a significance level of α = 0.05:

> **H0**: the mean hourly wage of all hourly workers is 30 USD.

> **H1**: the mean hourly wage of all hourly workers is different from 30 USD.

We compare one mean with a known value, so the fitting test is the one-sample t-test. Like in the lesson I only use a sample, in this case 100 hourly workers.

In [4]:
hourly = salaries[salaries['Salary or Hourly'] == "Hourly"]
hourly_sample = hourly['Hourly Rate'].sample(100, random_state=42)
st.ttest_1samp(hourly_sample, 30)


TtestResult(statistic=np.float64(2.6094408664329576), pvalue=np.float64(0.01047523509161658), df=np.int64(99))

**Answer:** The p-value is less then or equal *0.05* (alpha), that means we can reject the Null-Hypothesis. So the mean of the hourly rate of all hourly workers is significantly different than *30*. Whats the mean actually?

In [5]:
hourly['Hourly Rate'].mean()

np.float64(32.78855771628024)

The mean of all 8,022 hourly workers is about 32.79 USD and the mean of the sample is about 33.46 USD. Both are above 30, so the decision of the test holds up. Like in the lesson, normally you only have the sample, but here we can check it against all the data.

# Challenge 3 - Constructing Confidence Intervals

While testing our hypothesis is a great way to gather empirical evidence for accepting or rejecting the hypothesis, another way to gather evidence is by creating a confidence interval. A confidence interval gives a range of plausible values for the true mean of the population. A 95% confidence interval comes from a method that, over many repeated samples, captures the true mean 95% of the time.

To read more about confidence intervals, click [here](https://en.wikipedia.org/wiki/Confidence_interval).


In the cell below, we will construct a 95% confidence interval for the mean hourly wage of all hourly workers. 

The confidence interval is computed in SciPy using the `t.interval` function. You can read more about this function [here](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html).

To compute the confidence interval of the hourly wage, use the 0.95 for the confidence level, number of rows - 1 for degrees of freedom, the mean of the sample for the location parameter (`loc`) and the standard error for the scale (`scale`). The standard error can be computed using [this](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.sem.html) function in SciPy.

In [6]:
hourly_rate = hourly['Hourly Rate']
n = len(hourly_rate)
mean = hourly_rate.mean()
se = sem(hourly_rate)

ci = t.interval(0.95, df=n - 1, loc=mean, scale=se)
print(f"n={n}, mean={round(mean, 2)}, SE={round(se, 2)}, 95% CI = ({round(ci[0], 2)}, {round(ci[1], 2)})")

n=8022, mean=32.79, SE=0.14, 95% CI = (32.52, 33.05)


In [7]:
# the same interval, but only with the sample of 100 from Challenge 2
t.interval(0.95, df=len(hourly_sample) - 1, loc=hourly_sample.mean(), scale=sem(hourly_sample))

(np.float64(30.827892373813775), np.float64(36.08270762618624))

**Answer:** The 95% confidence interval for the mean hourly wage goes from about 32.52 to 33.05 USD. 30 is not in the interval, so 30 is not a plausible value for the true mean. That fits the result of Challenge 2.

The interval is that narrow because it uses all 8,022 hourly workers. With the sample of 100 it goes from about 30.83 to 36.08 USD, which is about 10 times wider. That is because the standard error gets smaller with a bigger sample size (it is the standard deviation divided by the square root of n).

# Challenge 4 - Hypothesis Tests of Proportions

Another type of one sample test is a hypothesis test of proportions. In this test, we examine whether the proportion of a group in our sample is significantly different than a fraction. 

You can read more about one sample proportion tests [here](http://sphweb.bumc.bu.edu/otlt/MPH-Modules/BS/SAS/SAS6-CategoricalData/SAS6-CategoricalData2.html).

In the cell below, use the `proportions_ztest` function from `statsmodels` to perform a hypothesis test that will determine whether the proportion of hourly workers in the City of Chicago is significantly different from 25%, at the 5% significance level.

First the hypotheses again, at a significance level of α = 0.05:

> **H0**: the proportion of hourly workers is 25%.

> **H1**: the proportion of hourly workers is different from 25%.

In [8]:
# count = hourly workers, nobs = all employees, value = the proportion from H0
# the test is two-sided by default
count = len(hourly)
nobs = len(salaries)

print(f"proportion of hourly workers = {round(count / nobs, 4)}")
proportions_ztest(count, nobs, value=0.25)

proportion of hourly workers = 0.2418


(np.float64(-3.5099964213703005), np.float64(0.00044811272490579676))

**Answer:** The p-value is about 0.00045, which is below *0.05* (alpha), so we can reject the Null-Hypothesis. The proportion of hourly workers (about 24.2%) is significantly different from 25%. The test statistic is negative (about -3.51), so the proportion is below 25%, not above it.

The difference itself is small though: less than one percentage point. With 33,183 employees even a difference like that becomes significant.

One limitation for all the challenges: the dataset already contains all current employees, so 24.2% and the mean of 32.79 are the actual values at that moment. The tests and the confidence interval only make sense if we treat this snapshot as a sample of a bigger population.